# RAG Agent with Google ADK + FAISS
**Department of Quantum Engineering — Retrieval-Augmented Generation Demo**

This notebook builds a RAG pipeline using:
- `sentence-transformers` for document embeddings
- `FAISS` for fast vector similarity search
- `Google ADK` (Agent Development Kit) with `gemini-2.0-flash` as the LLM

> **Before running:** Add your `GEMINI_API_KEY` to Colab Secrets (key icon in the left sidebar).

## 1. Install Dependencies

In [ ]:
!pip install -q google-adk google-generativeai sentence-transformers faiss-cpu numpy

## 2. Imports

In [ ]:
import os
import numpy as np
import faiss

from sentence_transformers import SentenceTransformer
from google.adk.agents import Agent
from google.adk.runners import Runner
from google.genai import types

## 3. API Key Setup

The cell below reads `GEMINI_API_KEY` from **Colab Secrets** (recommended).  
If you haven't added it yet: click the **key icon** in the left sidebar → **Add new secret** → name it `GEMINI_API_KEY`.

In [ ]:
try:
    from google.colab import userdata
    os.environ["GEMINI_API_KEY"] = userdata.get("GEMINI_API_KEY")
    print("API key loaded from Colab Secrets.")
except Exception:
    print("API key not found. Please set the GEMINI_API_KEY environment variable.")
   


## 4. Sample Department Documents

In [ ]:
documents = [
    # Department overview
    "The Department of Quantum Engineering (DQE) has around 140 students and 20 professors, focusing on applied quantum computing and intelligent systems.",
    "Students can choose from 5 courses ranging from core subjects to electives and hands-on project work, with strong industry exposure.",
    "DQE offers a postgraduate module 'Foundations of Quantum AI' and an undergraduate elective 'Quantum Machine Learning' using IBM Qiskit and PennyLane.",
    "All students have access to cloud quantum hardware through IBM Quantum Network and Amazon Braket as part of their coursework.",
    # Quantum AI research
    "DQE's primary research focus is Quantum AI — the intersection of quantum computing and artificial intelligence.",
    "The Quantum AI Lab investigates Quantum Neural Networks (QNNs) using parameterized quantum circuits (PQCs) and collaborates with a national quantum computing centre on 20-qubit and 50-qubit processors.",
    "A key research thread is Quantum Reinforcement Learning (QRL), where quantum agents learn policies faster than classical counterparts on specific problem classes.",
    "Project QuLearn is DQE's flagship initiative building hybrid classical-quantum models for drug discovery and materials science."
]

print(f"Loaded {len(documents)} documents.")

## 5. Create Embeddings

In [ ]:
embedding_model = SentenceTransformer("all-MiniLM-L6-v2")
doc_embeddings = embedding_model.encode(documents)

print(f"Embedding shape: {doc_embeddings.shape}")

## 6. Build FAISS Index

In [ ]:
dimension = doc_embeddings.shape[1]
index = faiss.IndexFlatL2(dimension)
index.add(np.array(doc_embeddings))

print(f"FAISS index built with {index.ntotal} vectors of dimension {dimension}.")

## 7. Retrieval Function (RAG Tool)

In [ ]:
def retrieve_context(query: str, top_k: int = 2) -> str:
    query_embedding = embedding_model.encode([query])
    distances, indices = index.search(np.array(query_embedding), top_k)
    retrieved_docs = [documents[i] for i in indices[0]]
    return "\n".join(retrieved_docs)

## 8. Google ADK Agent Setup

In [ ]:
rag_agent = Agent(
    model="gemini-2.0-flash",
    name="quantum_rag_agent",
    instruction="""
You are a RAG assistant for the Department of Quantum Engineering.

Answer ONLY using the provided context.

If information is unavailable, say:
'I could not find this in the department documents.'
"""
)

runner = Runner(agent=rag_agent)
print("Agent and runner initialised.")

## 9. Ask a Question

In [ ]:
user_query = "What research happens in DQE?"

context = retrieve_context(user_query)
print("--- Retrieved Context ---")
print(context)
print("-------------------------")

In [ ]:
prompt = f"""
Context:
{context}

Question:
{user_query}

Answer using only the context.
"""

response = runner.run(
    user_id="student_1",
    session_id="session_1",
    new_message=types.Content(
        role="user",
        parts=[types.Part(text=prompt)]
    )
)

print("--- Agent Response ---")
print(response)

## 10. Try Your Own Query

In [ ]:
my_query = "What courses does DQE offer?"  # ← change this

context = retrieve_context(my_query)
prompt = f"""
Context:
{context}

Question:
{my_query}

Answer using only the context.
"""

response = runner.run(
    user_id="student_1",
    session_id="session_2",
    new_message=types.Content(
        role="user",
        parts=[types.Part(text=prompt)]
    )
)

print("--- Agent Response ---")
print(response)